# ImageNet Benchmark — Baselines

Treina ResNet-50, EfficientNet-B0, MobileNetV3 em CIFAR-100 (proxy)

**Tempo esperado:**
- A100: ~30 min
- V100: ~45 min
- T4: ~60 min


In [ ]:
# Setup
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as T
import torchvision.models as models
from torchvision.datasets import CIFAR100
from torch.utils.data import DataLoader
import json
import os
import time
import numpy as np
from tqdm import tqdm

# Verifica GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
RESULTS_DIR = '/content/drive/My Drive/dNATY_Results'
os.makedirs(RESULTS_DIR, exist_ok=True)

print("\n✅ Setup pronto")

In [ ]:
# Download CIFAR-100 (proxy para ImageNet)
print("Baixando CIFAR-100...")

train_transforms = T.Compose([
    T.RandomCrop(32, padding=4),
    T.RandomHorizontalFlip(),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    T.RandomRotation(10),
    T.ToTensor(),
    T.Normalize((0.5071, 0.4867, 0.4408), (0.2675, 0.2565, 0.2761))
])

val_transforms = T.Compose([
    T.ToTensor(),
    T.Normalize((0.5071, 0.4867, 0.4408), (0.2675, 0.2565, 0.2761))
])

train_dataset = CIFAR100('/tmp/cifar100', train=True, download=True, transform=train_transforms)
val_dataset = CIFAR100('/tmp/cifar100', train=False, download=True, transform=val_transforms)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=256, shuffle=False, num_workers=2, pin_memory=True)

print(f"✅ Train: {len(train_loader)} batches")
print(f"✅ Val: {len(val_loader)} batches")

In [ ]:
# Define baselines
class ResNet50(nn.Module):
    def __init__(self, num_classes=100):
        super().__init__()
        self.model = models.resnet50(pretrained=False)
        self.model.fc = nn.Linear(2048, num_classes)

    def forward(self, x):
        return self.model(x)

    def count_params(self):
        return sum(p.numel() for p in self.parameters())


class EfficientNetB0(nn.Module):
    def __init__(self, num_classes=100):
        super().__init__()
        self.model = models.efficientnet_b0(pretrained=False)
        self.model.classifier[1] = nn.Linear(1280, num_classes)

    def forward(self, x):
        return self.model(x)

    def count_params(self):
        return sum(p.numel() for p in self.parameters())


class MobileNetV3Large(nn.Module):
    def __init__(self, num_classes=100):
        super().__init__()
        self.model = models.mobilenet_v3_large(pretrained=False)
        self.model.classifier[1] = nn.Linear(1280, num_classes)

    def forward(self, x):
        return self.model(x)

    def count_params(self):
        return sum(p.numel() for p in self.parameters())


print("✅ Modelos definidos")

In [ ]:
def evaluate(model, val_loader, device):
    """Avalia acurácia no conjunto de validação"""
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            outputs = model(images)
            _, preds = outputs.max(1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return correct / total


def train_model(model, name, epochs=25, lr=0.1):
    """Treina modelo com SAM + data augmentation"""
    print(f"\nTreinando {name}...")
    model = model.to(device)
    opt = optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=1e-4)
    crit = nn.CrossEntropyLoss(label_smoothing=0.1)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=lr*0.01)

    rho = 0.05  # SAM radius
    best_acc = 0

    t0 = time.time()

    for epoch in range(epochs):
        model.train()

        for images, labels in tqdm(train_loader, desc=f"{name} Epoch {epoch+1}/{epochs}", leave=False):
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            # SAM Step 1
            opt.zero_grad(set_to_none=True)
            loss = crit(model(images), labels)
            loss.backward()

            # SAM Step 2 - Perturbação
            with torch.no_grad():
                grad_norm = sum(p.grad.norm().pow(2) for p in model.parameters() if p.grad is not None).sqrt()
                scale = rho / (grad_norm + 1e-12)
                for p in model.parameters():
                    if p.grad is not None:
                        p.data_orig = p.data.clone()
                        p.data.add_(p.grad, alpha=scale)

            # SAM Step 3 - Recomputar
            opt.zero_grad(set_to_none=True)
            loss_perturbed = crit(model(images), labels)
            loss_perturbed.backward()

            # SAM Step 4 - Restaurar
            with torch.no_grad():
                for p in model.parameters():
                    if hasattr(p, 'data_orig'):
                        p.data.copy_(p.data_orig)

            opt.step()

        scheduler.step()

        acc = evaluate(model, val_loader, device)
        best_acc = max(best_acc, acc)
        print(f"  Epoch {epoch+1}/{epochs} | Acc: {acc*100:6.2f}% | Best: {best_acc*100:6.2f}%")

    elapsed = time.time() - t0
    final_acc = evaluate(model, val_loader, device)

    return {
        'accuracy': round(final_acc, 4),
        'best_accuracy': round(best_acc, 4),
        'params': model.count_params(),
        'time_s': round(elapsed, 1)
    }

print("✅ Funções de treino definidas")

---

## 🚀 TREINA OS BASELINES


In [ ]:
print("\n" + "="*70)
print("CIFAR-100 BASELINE BENCHMARK")
print("="*70)

baselines = {
    'ResNet-50': ResNet50(100),
    'EfficientNet-B0': EfficientNetB0(100),
    'MobileNetV3-Large': MobileNetV3Large(100),
}

results = {}
for name, model in baselines.items():
    result = train_model(model, name, epochs=25, lr=0.1)
    results[name] = result

print("\n" + "-"*70)
print("RESULTADOS FINAIS")
print("-"*70)
for name, metrics in results.items():
    print(f"{name:20} | Acc: {metrics['best_accuracy']*100:6.2f}% | Params: {metrics['params']:>10,} | Time: {metrics['time_s']:>6.0f}s")

In [ ]:
# Salva resultados
output = {
    'dataset': 'CIFAR-100 (proxy para ImageNet)',
    'baselines': results,
    'summary': {
        'best_model': max(results.items(), key=lambda x: x[1]['best_accuracy'])[0],
        'lightest_model': min(results.items(), key=lambda x: x[1]['params'])[0],
        'fastest_model': min(results.items(), key=lambda x: x[1]['time_s'])[0],
    },
    'timestamp': time.strftime("%Y-%m-%d %H:%M:%S")
}

out_path = os.path.join(RESULTS_DIR, 'cifar100_baseline_results.json')
with open(out_path, 'w') as f:
    json.dump(output, f, indent=2)

print(f"\n✅ Resultados salvos em: {out_path}")

print("\n" + "="*70)
print("RESUMO")
print("="*70)
for key, value in output['summary'].items():
    print(f"  {key}: {value}")

---

## 📊 Análise de Trade-offs


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

names = list(results.keys())
accs = [results[n]['best_accuracy']*100 for n in names]
params = [results[n]['params']/1e6 for n in names]  # em milhões

# Accuracy vs Parameters
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']
axes[0].scatter(params, accs, s=500, c=colors, alpha=0.7, edgecolors='black', linewidth=2)
for i, name in enumerate(names):
    axes[0].annotate(name, (params[i], accs[i]), xytext=(5, 5), textcoords='offset points', fontsize=10)
axes[0].set_xlabel('Parâmetros (Milhões)', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Acurácia CIFAR-100 (%)', fontsize=11, fontweight='bold')
axes[0].set_title('Accuracy vs Model Size', fontsize=12, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Accuracy vs Training Time
times = [results[n]['time_s']/60 for n in names]  # em minutos
axes[1].scatter(times, accs, s=500, c=colors, alpha=0.7, edgecolors='black', linewidth=2)
for i, name in enumerate(names):
    axes[1].annotate(name, (times[i], accs[i]), xytext=(5, 5), textcoords='offset points', fontsize=10)
axes[1].set_xlabel('Training Time (minutes)', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Acurácia CIFAR-100 (%)', fontsize=11, fontweight='bold')
axes[1].set_title('Accuracy vs Training Speed', fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'baseline_tradeoffs.png'), dpi=150, bbox_inches='tight')
plt.show()

print("\n✅ Gráficos salvos")